In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Example using One machine with multiple GPUs

# Check if CUDA is available and number of GPUs
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Check number of GPUs available
n_gpus = torch.cuda.device_count()
print(f"Number of GPUs available: {n_gpus}")

# Simple network to train
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(10, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Create dummy data for training
X = torch.randn(1000, 10)  # 1000 samples, 10 features
y = torch.randn(1000, 1)   # 1000 labels

dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

model = SimpleNN()

# If multiple GPUs are available, use DataParallel to distribute the model across GPUs
if n_gpus > 1:
    print("Using 2 GPUs")
    model = nn.DataParallel(model, device_ids=[0, 1])  # Assuming 2 GPUs are available

# Move the model to the GPU(s)
model = model.to(device)


criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.001)

# Training loop
num_epochs = 100
for epoch in range(num_epochs):
    model.train()  # Set the model to training mode
    running_loss = 0.0
    
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)  # Move data to the same device as the model
        
        optimizer.zero_grad()  # Zero the gradients
        
        outputs = model(inputs)  # Forward pass
        loss = criterion(outputs, labels)  # Compute the loss
        
        loss.backward()  # Backpropagation
        optimizer.step()  # Update the model parameters
        
        running_loss += loss.item()

    print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {running_loss / len(dataloader):.4f}")

print("Training complete!")


Using device: cuda
Number of GPUs available: 2
Using 2 GPUs
Epoch [1/100], Loss: 1.1056
Epoch [2/100], Loss: 1.0671
Epoch [3/100], Loss: 1.0905
Epoch [4/100], Loss: 1.0932
Epoch [5/100], Loss: 1.0611
Epoch [6/100], Loss: 1.1051
Epoch [7/100], Loss: 1.0622
Epoch [8/100], Loss: 1.0941
Epoch [9/100], Loss: 1.0724
Epoch [10/100], Loss: 1.0594
Epoch [11/100], Loss: 1.1066
Epoch [12/100], Loss: 1.0776
Epoch [13/100], Loss: 1.0616
Epoch [14/100], Loss: 1.0604
Epoch [15/100], Loss: 1.0670
Epoch [16/100], Loss: 1.0754
Epoch [17/100], Loss: 1.0946
Epoch [18/100], Loss: 1.0670
Epoch [19/100], Loss: 1.0683
Epoch [20/100], Loss: 1.0667
Epoch [21/100], Loss: 1.0669
Epoch [22/100], Loss: 1.0880
Epoch [23/100], Loss: 1.0772
Epoch [24/100], Loss: 1.0474
Epoch [25/100], Loss: 1.0793
Epoch [26/100], Loss: 1.0660
Epoch [27/100], Loss: 1.0773
Epoch [28/100], Loss: 1.0877
Epoch [29/100], Loss: 1.0520
Epoch [30/100], Loss: 1.0786
Epoch [31/100], Loss: 1.0521
Epoch [32/100], Loss: 1.0713
Epoch [33/100], Loss: